In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [2]:
# Cargar el dataset desde la subcarpeta data
df = pd.read_csv("data/Telco-Customer-Churn.csv")

# Eliminar identificador no relevante para predicción
df = df.drop(columns=["customerID"])

# Manejo de espacios en blanco en TotalCharges a NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")

# Mostrar porcentaje de nulos
porcentaje_nulos = (df["TotalCharges"].isnull().sum() / len(df)) * 100
print(f"Porcentaje de nulos en TotalCharges: {porcentaje_nulos:.3f}%")

# Codificar variable objetivo binaria
df["Churn"] = (df["Churn"] == "Yes").astype(int)

df.head()

Porcentaje de nulos en TotalCharges: 0.156%


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [3]:
# Separación de variables predictoras y objetivo
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Partición antes de cualquier imputación o transformación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Entrenamiento: {X_train.shape[0]} registros")
print(f"Prueba: {X_test.shape[0]} registros")

Entrenamiento: 5634 registros
Prueba: 1409 registros


In [4]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [col for col in X.columns if col not in num_cols]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [5]:
modelos = {
    "Regresion Logistica": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
}

mejor_auc = 0
mejor_pipeline = None
mejor_nombre = ""

for nombre, clasificador in modelos.items():
    pipe = Pipeline(steps=[
        ("preprocesador", preprocessor),
        ("clasificador", clasificador)
    ])
    
    pipe.fit(X_train, y_train)
    
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probs)
    
    print(f"========== {nombre} ==========")
    print(f"ROC-AUC: {auc:.4f}")
    print(classification_report(y_test, preds))
    
    if auc > mejor_auc:
        mejor_auc = auc
        mejor_pipeline = pipe
        mejor_nombre = nombre

print(f"Mejor modelo: {mejor_nombre} (ROC-AUC: {mejor_auc:.4f})")

========== Regresion Logistica ==========
ROC-AUC: 0.8419
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

========== Random Forest ==========
ROC-AUC: 0.8426
              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1035
           1       0.67      0.51      0.58       374

    accuracy                           0.80      1409
   macro avg       0.76      0.71      0.73      1409
weighted avg       0.79      0.80      0.80      1409

Mejor modelo: Random Forest (ROC-AUC: 0.8426)


In [6]:
os.makedirs("models", exist_ok=True)

# Guardar el pipeline completo listo para inferencia sobre datos crudos
joblib.dump(mejor_pipeline, "models/churn_model_pipeline.joblib")
print("Modelo guardado exitosamente en: models/churn_model_pipeline.joblib")

Modelo guardado exitosamente en: models/churn_model_pipeline.joblib
